In [1]:
import warnings
from pathlib import Path
import pickle as pkl

import prism

from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
)
from imagematerials.preprocessing import get_preprocessing_data


In [2]:
vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"),
                                    cache="prep_vema.nc",
                                    climate_policy_scenario_dir = None, 
                                    circular_economy_scenario_dirs = None)


In [3]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [4]:
factory = ModelFactory(
    vhc_sector, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).add(Maintenance
    )
model = factory.finish()

In [5]:
pkl_file = Path("reporting.pkl")
if pkl_file.is_file():
    with open(pkl_file, "rb") as handle:
        model = pkl.load(handle)
else:
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)
    model.save_pkl(pkl_file)

In [8]:
model["vehicles"]["stocks"]

Magnitude,[[[0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] ... [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0]] [[0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] ... [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0]] [[0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] ... [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0]] ... [[5761445.833333333 47915958.333333336 22945916.666666668 ... 7502116.666666667 76870250.0 52110333.333333336] [85.69086361495962 2506.706354639835 126.68602425582432 ... 592.7962400523337 368.1178504755299 31.720176697705377] [4976.465097939773 46703.85054328995 1826.8105907071529 ... 2649.528021533531 712.8602012438599 7274.716889044378] ... [27627.56741701439 529752.4692318751 201800.43581703768 ... 12878.809392906534 609129.8685455509 335122.33514841297] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.6561021447990458 1.0454480829599904 0.5928845985431768 ... 1.2078016814768977 4.511541808683633 4.79719495445806]] [[5817275.0 48457625.0 23042887.5 ... 7588683.333333333 76382375.0 51994958.333333336] [85.63335192220968 2504.465892354522 126.44810664340481 ... 592.1460773794169 372.56012220116395 32.442314708595134] [5007.538172351757 46833.77904134958 1838.043221558687 ... 2667.606655340677 722.7978422717583 7427.411309794323] ... [27403.21077859636 521479.15953819896 195924.37910657193 ... 12343.68855705891 573743.9391852507 319979.8285828688] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.6620636095881828 1.2302648580709834 0.680927398519762 ... 1.2010087709703232 5.0262314637586245 4.985761308007438]] [[5873304.166666667 49004375.0 23137183.333333332 ... 7675729.166666667 75972708.33333333 51920333.333333336] [85.57186584418838 2502.0150214371974 126.1849399951439 ... 591.6610465180676 376.65779134503606 33.11739911503717] [5038.432281890381 46957.46365382752 1849.5963983833956 ... 2686.7208992150026 732.826397455324 7580.9902147183875] ... [27100.04055980486 512652.32694242307 189776.22150067159 ... 11804.682286355044 537802.7868728914 304249.9106583887] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.6689016004895418 1.4399778295451953 0.7764574403708978 ... 1.1941016984071782 5.557025259920483 5.178181258208292]]]
Units,count


In [15]:
from imagematerials.concepts import create_vehicle_graph, Node

In [19]:
kgraph = create_vehicle_graph()
kgraph.add(Node("Rail", inherits_from="Vehicles"))
kgraph["Freight Trains"].synonyms.append("Rail|Cargo")
kgraph["Freight Trains"].inherits_from = "Rail"
kgraph.add(Node("Rail|Passenger", inherits_from="Rail"))
for subtype in ["Trains", "High Speed Trains"]:
    kgraph[subtype].inherits_from = "Rail|Passenger"

kgraph.find_one_relation(["Freight Trains", "Trains", "High Speed Trains"], "Rail|Cargo")

['Freight Trains']

In [23]:
data = model["vehicles"]["stocks"].sel(Time=range(2005, 2105, 5))

In [24]:
kgraph.aggregate_sum(data, ["Rail", "Rail|Passenger", "Rail|Cargo"])

Magnitude,[[[1854.8015565688017 18236.6971901296 738.1386422883634 ... 2096.3432970473978 2420.022557510137 545.3314446315092] [1768.7076778750672 19002.121699201132 702.8383831257623 ... 2026.4164402944666 1509.3848020383516 506.3789062912484] [1845.8560314350243 20223.983393620536 724.3308584273374 ... 2227.2711458734507 1135.1882808161135 528.590790548481] ... [6043.553440781751 56023.12910777404 3836.1857159029973 ... 4776.553746142316 11619.241074615422 10515.922088679665] [6455.164924267323 58641.180667828834 4443.912002871111 ... 4968.98319556776 11755.24513833022 11965.009224611003] [6892.099962033489 61563.22598078634 5237.289404792256 ... 5178.13669333568 12280.52716947271 13467.010161048634]] [[104.47021254318919 954.9343090992458 283.9762220240524 ... 1534.1852000432912 2373.4470844360044 278.85958863307326] [91.04323102870087 970.7664345994503 226.4905226803747 ... 1427.603473788182 1454.7824044182537 181.8356532454578] [96.27395367921432 1176.4973940561388 214.94051456356496 ... 1563.0005440664818 1071.2202695813376 141.0445844050172] ... [1343.859507992389 10414.686761348261 2106.284206420768 ... 2273.6385212622263 10986.13780205374 4462.949075301449] [1574.6154536554627 12313.196128567528 2652.275566073925 ... 2371.039213732186 11071.89934676129 5143.8179590415475] [1853.6676801431074 14605.762326958818 3387.693006408861 ... 2491.4157941206777 11547.700772017386 5886.019946330247]] [[1750.3313440256125 17281.762881030354 454.16242026431104 ... 562.1580970041067 46.57547307413277 266.47185599843596] [1677.6644468463664 18031.355264601683 476.34786044538754 ... 598.8129665062847 54.602397620097854 324.54325304579066] [1749.5820777558101 19047.485999564396 509.3903438637724 ... 664.2706018069689 63.968011234776014 387.5462061434638] ... [4699.693932789362 45608.44234642578 1729.9015094822294 ... 2502.9152248800906 633.1032725616819 6052.973013378216] [4880.54947061186 46327.984539261306 1791.6364367971864 ... 2597.943981835574 683.3457915689294 6821.191265569456] [5038.432281890381 46957.46365382752 1849.5963983833956 ... 2686.7208992150026 732.826397455324 7580.9902147183875]]]
Units,count
